In [0]:
# configure paths
catalog = "global_mart_retail_dev"
bronze_table = "superstore_orders"
silver_table = "sales"

# table path
bronze_path = f"{catalog}.bronze.{bronze_table}"
silver_path = f"{catalog}.silver.{silver_table}"

print(f"Bronze table path: {bronze_path}")
print(f"Silver table path: {silver_path}")

In [0]:
# import libraries
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# read bronze data 
bronze_df = spark.read.table(bronze_path)#.filter(col("ingestion_timestamp") >= expr("current_timestamp() - INTERVAL 72 HOURS"))

# creating silver table for orders 
df_clean_sales = (         
    bronze_df.select(
    trim(col("Row_ID")).cast("int").alias("sales_id"),
    trim(col("Order_ID")).alias("order_id"),
    upper(trim(col("Customer_ID"))).alias("customer_id"),
    trim(col("Product_ID")).alias("product_id"),
    trim(col("Postal_Code")).alias("postal_code"),
    trim(col("Region")).alias("region"),
    col("Order_Date").alias("order_date"),
    col("Ship_Date").alias("ship_date"),
    trim(col("Ship_Mode")).alias("ship_mode"), 
    trim(col("Sales")).cast("decimal(18,4)").alias("sales"),
    trim(col("Quantity")).cast("int").alias("quantity"),
    trim(col("Discount")).cast("double").alias("discount"),
    trim(col("Profit")).cast("decimal(18,4)").alias("profit")
                      
).dropDuplicates(["sales_id"]))


display(df_clean_sales.limit(5))


print(f"total count: {df_clean_sales.count()}") # count after dedeuplication


display(df_clean_sales.limit(5))



In [0]:
df_clean_sales = df_clean_sales.filter(
    col("sales_id").isNotNull() & 
    col("order_id").isNotNull() & 
    col("customer_id").isNotNull() & 
    col("product_id").isNotNull()
)


print(f"total count before filters: {df_clean_sales.count()}")

df_clean_sales = df_clean_sales.filter(col("quantity") >= 0)
print(f"After removing negative quantity: {df_clean_sales.count()}")

#df_clean_sales = df_clean_sales.filter(col("ship_date") >= col("order_date"))
#print(f"After removing invalid dates: {df_clean_sales.count()}")

#df_clean_sales = df_clean_sales.filter(col("sales") > 0)
#print(f"After removing non-positive sales: {df_clean_sales.count()}")

In [0]:
# Create the silver sales table if it does not exist
spark.sql(
    """
        CREATE TABLE IF NOT EXISTS global_mart_retail_dev.silver.sales
        (
            sales_id  INT not null,
            order_id string not null,
            customer_id string not null,
            product_id string not null,
            postal_code string ,
            region string,
            order_date date,
            ship_date date ,
            ship_mode string,
            sales decimal (18,4),
            discount double,
            quantity int,
            profit decimal (18,4),
            load_timestamp timestamp not null

        )
        USING DELTA
    """
)

df_clean_sales = df_clean_sales.withColumn("load_timestamp", current_timestamp())


# MERGE UPSERT pattern

target_table = DeltaTable.forName(spark, "global_mart_retail_dev.silver.sales")

target_table.alias("target").merge(
    df_clean_sales.alias("source"),
    "target.sales_id = source.sales_id"  # Match on primary key
).whenMatchedUpdateAll(  # Update existing records
).whenNotMatchedInsertAll(  # Insert new records
).execute()


In [0]:
%sql
select count(*) from global_mart_retail_dev.silver.sales

In [0]:
# Log data quality metrics
print(f"Total records processed: {bronze_df.count()}")
print(f"After deduplication: {df_clean_sales.count()}")
print(f"Records with negative quantity (excluded): {bronze_df.filter(col('Quantity') < 0).count()}")
print(f"Records with NULL keys (excluded): {bronze_df.filter(col('Row_ID').isNull() | col('Order_ID').isNull()).count()}")

In [0]:
%sql
select * from global_mart_retail_dev.silver.sales